In [1]:
import os
import json
import pandas as pd
from maomao.utils.constants import *
from maomao.parsing.metadata_utils import *
from maomao.parsing.integrated_dataset_utils import *

#### Embryotoxic dataset integration and label-consistency analysis
- This notebook performs multi-source integration, quality control, and label-consistency analysis for embryotoxic peptide annotations by aggregating data from several independent databases.

- The inputs consist of embryotoxic peptide datasets curated from multiple sources, including BIOPEP-UWM, CICERON, and MultiPep. Each source provides sequence-level embryotoxic annotations with potentially overlapping coverage and heterogeneous labeling completeness.

- All peptide sequences associated with embryotoxic activity are first pooled across sources to construct a global set of unique sequences. A pivot table is then generated in which each row corresponds to a unique peptide sequence and each column represents a data source, enabling direct comparison of embryotoxic annotations across databases.

- Sequence-level quality control is applied prior to label integration. Peptides containing non-canonical amino acids are removed, and length-based filtering is enforced using globally defined minimum and maximum sequence length thresholds. These steps ensure biochemical validity and consistency across sources.

- After filtering, source-specific embryotoxic labels are mapped onto the pivot table using a standardized encoding scheme that distinguishes positive, negative, unlabeled, and unknown annotations. Label agreement across sources is quantified by computing per-sequence label counts, positive and negative vote percentages, and a set of high-level classification flags that identify sequences with consistent evidence (exclusive positive or exclusive negative), sequences with no definitive labels, and sequences with conflicting annotations.

- Ambiguous sequences are explicitly retained and further stratified according to the proportion of positive annotations, allowing flexible downstream handling based on confidence thresholds.

- The notebook produces curated, non-overlapping subsets of embryotoxic peptides, including strictly embryotoxic sequences, strictly non-embryotoxic sequences, and ambiguous sequences with mixed evidence. In addition, a comprehensive metadata file is generated, summarizing sequence filtering statistics, length distributions, source contributions, and label agreement patterns.

In [2]:
name_task = "toxic_effect_classification"
output_folder = "../../processed_data/integrating_and_cleaning_data/embryotoxic"

# PATH_EXPORT are imported from peptide_toxicity_classifier.constants.
# Update them in constants.py according to the required input and export paths.

- Reading all sources

In [3]:
df_BIOPEP_UWM_embryotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/BIOPEP-UWM/processed_embryotoxic_dataset.csv")
df_BIOPEP_UWM_embryotoxic = df_BIOPEP_UWM_embryotoxic.rename(columns={"label": "embryotoxic"})

In [4]:
df_CICERON_embryotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/CICERON/processed_embryotoxic_dataset.csv")
df_CICERON_embryotoxic = df_CICERON_embryotoxic.rename(columns={"label": "embryotoxic"})

In [5]:
df_MultiPep_embryotoxic = pd.read_csv(f"{PATH_EXPORT}/{name_task}/MultiPep/processed_embryotoxic_dataset.csv")
df_MultiPep_embryotoxic = df_MultiPep_embryotoxic.rename(columns={"label": "embryotoxic"})

- Collecting all sequences for activity

In [6]:
df_list_embryotoxic = [
    df_BIOPEP_UWM_embryotoxic, df_CICERON_embryotoxic, df_MultiPep_embryotoxic
]
unique_sequence_embryotoxic = count_unique_sequence(df_list_embryotoxic)

3


- Create pivote dataset

In [7]:
df_pivote = create_pivote(unique_sequence_embryotoxic)

- Removing sequences with non canonical residues 

In [8]:
n_before_canon = df_pivote.shape[0]
df_pivote["is_canon"] = df_pivote["sequence"].apply(check_sequence)
n_after_canon = df_pivote[df_pivote["is_canon"]].shape[0]

In [9]:
print(df_pivote["is_canon"].value_counts())
df_pivote = df_pivote[df_pivote["is_canon"]]

is_canon
True    3
Name: count, dtype: int64


- Filter sequences by length

In [10]:
df_pivote["length"] = df_pivote["sequence"].str.len()
df_pivote["length"].describe()

count    3.000000
mean     4.333333
std      1.154701
min      3.000000
25%      4.000000
50%      5.000000
75%      5.000000
max      5.000000
Name: length, dtype: float64

In [11]:
n_before_length = n_after_canon
df_pivote["filter_length"] = df_pivote["length"].apply(check_length)
n_after_length = df_pivote[df_pivote["filter_length"]].shape[0]

In [12]:
df_pivote["filter_length"].value_counts()

filter_length
True     2
False    1
Name: count, dtype: int64

In [13]:
length_series = df_pivote[df_pivote["filter_length"]]["length"]

length_dist = {
    "min": length_series.min(),
    "max": length_series.max(),
    "mean": length_series.mean(),
    "median": length_series.median()
}

In [14]:
df_pivote = df_pivote[df_pivote["filter_length"]]
df_pivote.shape

(2, 4)

In [15]:
df_pivote = df_pivote.drop(columns=["is_canon", "filter_length", "length"])

In [16]:
df_list_embryotoxic = [("BIOPEP-UWM", df_BIOPEP_UWM_embryotoxic),
                     ("CICERON", df_CICERON_embryotoxic),
                     ("MultiPep", df_MultiPep_embryotoxic)
                ]

In [17]:
for source, dataset in df_list_embryotoxic:
    dataset = dataset[["sequence", "embryotoxic"]]
    dataset = dataset.drop_duplicates(subset="sequence")
    mapping = dataset.set_index("sequence")["embryotoxic"]

    # Mapear sin explotar memoria
    df_pivote[source] = (
        df_pivote["sequence"]
        .map(mapping)
        .fillna(999)
        .astype("int16")
    )

- Working with pivote for detecting ambiguous sequences 

In [18]:
df_pivote = process_count_labels(df_pivote)  # Verify the consistency of the labels by source

In [19]:
df_pivote["negative"].value_counts() # Includes sources labeled as nevative (0) and unlabeled (2)

negative
False    2
Name: count, dtype: int64

In [20]:
df_pivote["exclusive_0"].value_counts()

exclusive_0
False    2
Name: count, dtype: int64

In [21]:
df_pivote["positive"].value_counts() # Includes sources labeled as positive (1) and unlabeled (2)

positive
True    2
Name: count, dtype: int64

In [22]:
df_pivote["exclusive_1"].value_counts()

exclusive_1
True    2
Name: count, dtype: int64

In [23]:
df_pivote["only_unlabel"].value_counts() # Includes only sources unlabeled (2)

only_unlabel
False    2
Name: count, dtype: int64

In [24]:
df_pivote.sort_values(by="percentage_1", ascending=False)

,sequence,BIOPEP-UWM,CICERON,MultiPep,counts_1,counts_0,counts_unlabel,counts_unknown,positive,negative,exclusive_1,exclusive_0,only_unlabel,percentage_0,percentage_1
1,IKVAV,1,1,1,3,0,0,0,True,False,True,False,False,0.0,100.0
2,YIGSR,1,1,1,3,0,0,0,True,False,True,False,False,0.0,100.0


- Splitting data into only negative, only positive, and with amiguous data

In [25]:
negative = df_pivote[df_pivote["negative"]]

In [26]:
only_negative = df_pivote[df_pivote["exclusive_0"]]

In [27]:
positive = df_pivote[df_pivote["positive"]]

In [28]:
only_positive = df_pivote[df_pivote["exclusive_1"]]

In [29]:
only_unlabel = df_pivote[df_pivote["only_unlabel"]]

In [30]:
df_ambiguous = df_pivote[(df_pivote["positive"] == False) & (df_pivote["negative"] == False) & (df_pivote["only_unlabel"] == False)]

- Processing ambiguous data

In [31]:
df_ambiguous = categorize_percentage(df_ambiguous)

In [32]:
df_ambiguous["Category_pbb"].value_counts()

Series([], Name: count, dtype: int64)

- Working with metada

In [33]:
seq_stats = {
    "canonical": {
        "before": n_before_canon,
        "after": n_after_canon
    },
    "length": {
        "before": n_before_length,
        "after": n_after_length,
        "min": MIN_LENGTH_SEQUENCE,
        "max": MAX_LENGTH_SEQUENCE
    },
    "length_dist": length_dist
}

metadata = build_dataset_metadata(
    task="embryotoxic",
    source_list=df_list_embryotoxic,
    pivote_df=df_pivote,
    outputs={
        "only_positive": only_positive,
        "only_negative": only_negative,
        "ambiguous": df_ambiguous
    },
    seq_stats=seq_stats,
    filters={
        "canonical_residues": True,
        "length_filter": True
    }
)
metadata

{'task': 'embryotoxic',
 'generated_at': '2026-09-04T20:36:12.165173',
 'sources': {'n_unique_sequences': {'BIOPEP-UWM': 3,
   'CICERON': 3,
   'MultiPep': 3}},
 'filters': {'canonical_residues': {'applied': True},
  'length_filter': {'applied': True, 'min_length': 5, 'max_length': 70}},
 'sequence_statistics': {'canonical_filter': {'before': 3, 'after': 3},
  'length_filter': {'before': 3, 'after': 2},
  'length_distribution': {'min': 5, 'max': 5, 'mean': 5.0, 'median': 5.0}},
 'statistics': {'total_sequences_final': 2,
  'positive': {'positive_and_unlabel': 2, 'only_positive': 2},
  'negative': {'negative_and_unlabel': 0, 'only_negative': 0},
  'only_unlabel': 0,
  'ambiguous': {'n_sequences': 0}}}

- Exporting data

In [34]:
os.makedirs(output_folder, exist_ok=True)

In [35]:
with open(f"{output_folder}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [36]:
positive.shape

(2, 15)

In [37]:
only_positive.shape

(2, 15)

In [38]:
negative.shape

(0, 15)

In [39]:
only_negative.shape

(0, 15)

In [40]:
only_unlabel.shape

(0, 15)

In [41]:
df_ambiguous.shape

(0, 16)

In [42]:
positive.to_csv(f"{output_folder}/positive.csv", index=False)